# Coloration black-box on **LA2A** (SignalTrain) — SOTA LSTM on the amplitude-matched signal

**Google Colab**: Runtime -> **GPU**. Open via *File -> Open notebook -> GitHub*
(`5aola/Virtual-Analogue-Compressor-Modelling`); cell 1 clones the repo for the
`08_la2a` + `06_output` modules and mounts Drive for the dataset. **Push local
changes before running.**

## What this is

A **retarget of [`06_output/train_lstm_color_blackbox.ipynb`](../06_output/train_lstm_color_blackbox.ipynb)**
— the coloration black-box (obedient gain + SOTA coloration trunk) — from
Diff-SSL-G-Comp to the **Teletronix LA-2A** (SignalTrain 1.1 dataset). The model
(`06_output/model_color_lstm.ColorBlackboxLSTM`), the training system
(`GainPriorSystem` with the exact 02b loss), and the TBPTT regime are the *same*
`06_output` code, unchanged. Only the dataset, its split, and the conditioning
width (2 LA2A knobs instead of 4) differ. Idea recap:

```
matched = x · 10^(gr/20)                    <- gain: explicit multiply, obeys
y = tanh( lin( LSTM(matched (+) tvcond) ) ) <- coloration: SOTA LSTM32TVC trunk
```

- **The only difference from the LA2A SOTA baseline
  ([`train_lstm_la2a_tvc.ipynb`](train_lstm_la2a_tvc.ipynb)) is the input signal**
  (matched instead of dry) — same trunk size, same loss family, same split, so
  the pair is a clean input-swap comparison on LA2A.
- **It never sees the GR value**, so there is nothing to disobey — any commanded
  GR trajectory, including one predicted by the LA2A blackbox GR predictor
  ([`train_lstm_blackbox_gr.ipynb`](train_lstm_blackbox_gr.ipynb)), is applied
  exactly (the sidechain goal).
- **No zero-init anchor.** The untrained model is NOT the amplitude match;
  cell 6 prints the amp-match L1 as the baseline the run must beat.
- Trained on the **oracle** GR (recomputed on-the-fly from each `(dry, wet)`
  crop); predicted/const GR behaviour is an inference-time swap, never trained on.
- **GR clamp re-verified on LA2A** (cell 6b): the amplitude-match clamp
  [-30, +5] dB was justified on diffssl (curves span ~[-28.3, +4.2] dB) — LA2A
  GR reaches [-49.6, +36.2] dB, but a full-file scan shows the clamp touches
  only 0.009% of energy-valid samples below (deep-GR tails at pr >= 85) and
  0.45% above (+GR = wet noise floor / session level mismatch, present even at
  pr=0 — not compression). Same range as gr_target.py, so the cascade's
  predicted GR stays in-range by construction.

## Dataset — SignalTrain LA2A (`data/LA2A/all/`)

84 **long recordings** (4-20 min each, 44.1 kHz mono float), one per
`(Comp/Limit, Peak Reduction)` setting: `input_<id>_.wav` (dry) +
`target_<id>_LA2A_<Nc>__<cl>__<pr>.wav` (wet). **2 knobs** (`la2a_info.ini`):
Comp/Limit in {0,1} switch, Peak Reduction in {0,5,..,100}. 42 unique settings.

**GR is recomputed on-the-fly** from each `(dry, wet)` crop with the *exact*
export function — `src.dsp_torch.gain_reduction_db(dry, wet, 1024)` (causal
zero-left-padded 1024-RMS). A 1023-sample lookback fills the causal window from
real preceding samples, so each crop's GR is **bit-identical to slicing the
full-file curve**. No `.pt` cache — only the WAVs are needed
([`08_la2a/dataset_la2a.py`](dataset_la2a.py), the same `(dry, gr, wet, params)`
contract the gain-prior WS notebook uses).

## Split — percentage-based, temporal within each recording (`08_la2a/splits_la2a.py`)

Diff-SSL's song-level / `test_ground_truth` policy does **not** transfer: every
LA2A setting lives in only one (a few in two-three) recording(s), so holding out
whole recordings would delete a setting from training and break the knob
conditioning. Instead we split **temporally by time fraction within each
recording** — the standard LA2A methodology (SignalTrain, Steinmetz TCN,
Comparative-Study, Optical-DRC all test on held-out *audio regions* at the same
settings):

```
per recording:  [0, 0.8) -> train    [0.8, 0.9) -> val    [0.9, 1.0) -> test
```

- **Every setting is in all three splits** -> conditioning fully learnable; the
  test set probes generalisation to *unseen audio at known settings*.
- **No content leakage** — boundaries are by time fraction, identical for every
  recording. Same crop budget as the other LA2A runs (`crops_per_pair` 24/4/8),
  so all LA2A experiments train/eval on the *same audio windows*.

## Training recipe — exact 02b loss + diffssl TBPTT, speed-tuned

Same as the diffssl coloration black-box: 3 s crops, state reset per batch,
TBPTT sub-steps of 4410, AdamW + cosine, fixed 100-epoch budget, bf16 autocast,
and the **exact 02b loss** (`0.5*L1 + 0.5*MR-STFT "sota"`; env/pe off — extra
terms would confound the input-swap comparison, and env is void anyway since
the multiply pins the envelope).

**Speed deviation from the 02b regime** (the sample-serial LSTM is
latency-bound, so batch size is nearly free): **batch 64 @ lr 2e-3** (02b: 16 @
1e-3; sqrt-scaled LR for the 4x batch). Model, loss, split, TBPTT sub-step and
epoch budget are unchanged. Restore `BATCH_SIZE = 16`, `LR = 1e-3` in cell 3
for a headline-number parity run.

Comparable runs: LA2A SOTA LSTM32TVC (same trunk, dry input,
[`train_lstm_la2a_tvc.ipynb`](train_lstm_la2a_tvc.ipynb)), LA2A oracle
`gain_prior_ws` ([`train_lstm_gain_prior_ws.ipynb`](train_lstm_gain_prior_ws.ipynb)),
and the raw amplitude match.

In [ ]:
# -- 0. Dependencies ---------------------------------------------------
# Uses nablafx (TVFiLMCond). Pin numpy first so lightning/nablafx installs
# can't downgrade Colab's numpy 2.x and break torch. Install lightning/nablafx
# --no-deps so they can't clobber Colab's CUDA torch. `rational` /
# `frechet_audio_distance` are nablafx import-chain deps we never use; stub
# both so `from nablafx...` doesn't drag in broken wheels.
!pip install -q "numpy>=2.0,<2.6"
!pip install -q torchmetrics soundfile auraloss einops lightning-utilities packaging
!pip install -q --no-deps lightning nablafx

import sys, types

rational = types.ModuleType("rational")
rational.torch = types.ModuleType("rational.torch")
rational.torch.Rational = type("Rational", (), {})
sys.modules["rational"], sys.modules["rational.torch"] = rational, rational.torch

fad = types.ModuleType("frechet_audio_distance")
fad.FrechetAudioDistance = type("FrechetAudioDistance", (), {})
sys.modules["frechet_audio_distance"] = fad

import numpy as np, torch
assert np.__version__.startswith("2."), f"numpy {np.__version__} - restart runtime, re-run cell 0"
print(f"numpy {np.__version__}, torch {torch.__version__}")

In [ ]:
# -- 1. Mount Drive (dataset) + clone repo from GitHub (code) ---------
# The repo is NOT synced to Drive (only data/ is). Code comes from GitHub -
# push local changes before (re)running this cell; re-running pulls updates.

import os
import sys
from pathlib import Path

from google.colab import drive

drive.mount("/content/drive", force_remount=False)

DRIVE_DATA_ROOT = "/content/drive/Othercomputers/MacBook Air/data/LA2A"
REPO_URL = "https://github.com/5aola/Virtual-Analogue-Compressor-Modelling.git"
REPO_ROOT = "/content/Virtual-Analogue-Compressor-Modelling"

if os.path.isdir(REPO_ROOT):
    !git -C "{REPO_ROOT}" fetch origin
    !git -C "{REPO_ROOT}" reset --hard origin/main
else:
    !git clone --depth 1 "{REPO_URL}" "{REPO_ROOT}"

DATA_ROOT = DRIVE_DATA_ROOT

# Module dirs: LA2A dataset/split (08_la2a) + the coloration model/system it
# reuses unchanged (06_output). Both go on sys.path.
LA2A_DIR = os.path.join(REPO_ROOT, "08_la2a")
MODEL_DIR = os.path.join(REPO_ROOT, "06_output")
assert os.path.isfile(os.path.join(LA2A_DIR, "dataset_la2a.py")), (
    f"Clone failed or stale: {LA2A_DIR}. Did you push local changes?"
)
assert os.path.isfile(os.path.join(MODEL_DIR, "model_color_lstm.py")), (
    f"Missing 06_output model modules: {MODEL_DIR}"
)

# Same runs dir as the LA2A gain-prior family (mirrors diffssl, where the
# coloration black-box lives in diffssl_gain_prior_runs).
OUTPUT_DIR = os.path.join(os.path.dirname(DATA_ROOT), "la2a_gain_prior_runs")

assert os.path.isdir(os.path.join(DATA_ROOT, "all")), (
    f"No all/ under {DATA_ROOT} - sync the SignalTrain LA2A WAVs to Drive first."
)
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Drop cached local modules so a prior run cannot keep stale classes.
for _name in list(sys.modules):
    if _name in ("dataset_la2a", "splits_la2a", "model_color_lstm",
                 "system_gainprior", "amplitude_match"):
        del sys.modules[_name]

# repo root (for `src` + `nablafx`) + both module dirs
for p in (REPO_ROOT, os.path.join(REPO_ROOT, "nablafx"), MODEL_DIR, LA2A_DIR):
    if p not in sys.path:
        sys.path.insert(0, p)

print(f"REPO_ROOT  : {REPO_ROOT}")
print(f"LA2A_DIR   : {LA2A_DIR}")
print(f"MODEL_DIR  : {MODEL_DIR}")
print(f"DATA_ROOT  : {DATA_ROOT}")
print(f"OUTPUT_DIR : {OUTPUT_DIR}")

In [ ]:
# -- 2. Cache dataset to Colab local SSD ------------------------------
# Only the input/target WAVs are cached (~29 GB float32). The 18 GB gr_curves/
# tree is deliberately NOT needed: dataset_la2a recomputes GR on-the-fly per
# crop, bit-identical to the exported .pt. One-time copy per session.

import shutil
from dataset_la2a import discover_la2a_pairs

LOCAL_DATA_ROOT = "/content/LA2A"
pairs = discover_la2a_pairs(DATA_ROOT)
print(f"Caching {len(pairs)} recordings (input+target WAVs) -> {LOCAL_DATA_ROOT}")

def _mirror(src, dst):
    src, dst = Path(src), Path(dst)
    if not dst.exists() or dst.stat().st_size != src.stat().st_size:
        dst.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(src, dst)

for i, p in enumerate(pairs, 1):
    for key in ("dry", "wet"):
        _mirror(p[key], Path(LOCAL_DATA_ROOT) / Path(p[key]).relative_to(DATA_ROOT))
    if i % 10 == 0 or i == len(pairs):
        print(f"  cached {i}/{len(pairs)}")

DATA_ROOT = LOCAL_DATA_ROOT
print(f"Using local cache: {DATA_ROOT}")

In [ ]:
# -- 3. Imports & hyper-parameters (coloration black-box + exact 02b loss) --

import importlib
import json
from datetime import datetime

import torch
import lightning as pl
from lightning.pytorch.callbacks import (
    LearningRateMonitor, ModelCheckpoint, TQDMProgressBar,
)
from lightning.pytorch.loggers import CSVLogger, TensorBoardLogger

import dataset_la2a as _dataset_la2a
importlib.reload(_dataset_la2a)
from dataset_la2a import (
    RMS_WINDOW, SAMPLE_LENGTH, SAMPLE_RATE,
    La2aCropDataModule, discover_la2a_pairs,
)  # BATCH_SIZE is set explicitly below (L4 speed tuning), not imported

import splits_la2a as _splits_la2a
importlib.reload(_splits_la2a)
from splits_la2a import (
    LA2A_PARAM_ORDER, LA2A_PARAM_RANGES, build_la2a_split_manifest,
)

import model_color_lstm as _model_color_lstm
importlib.reload(_model_color_lstm)
from model_color_lstm import ColorBlackboxLSTM

import system_gainprior as _system_gainprior   # reused UNCHANGED from 06_output
importlib.reload(_system_gainprior)
from system_gainprior import GainPriorSystem

print(torch.cuda.get_device_name(0) if torch.cuda.is_available() else "WARNING: CPU runtime")

# -- split: percentage-based, temporal within each recording --
SPLIT_SEED  = 42
TRAIN_FRAC, VAL_FRAC, TEST_FRAC = 0.8, 0.1, 0.1
# evenly-spaced crops per recording per split — same budget as the other LA2A
# runs, so all LA2A experiments train/eval on the exact same audio windows.
CROPS_PER_PAIR = {"train": 24, "val": 4, "test": 8}

# -- training (diffssl TBPTT regime, SPEED-TUNED — deviates from the exact 02b
# recipe, which used batch 16 @ lr 1e-3. Batch 64 covers 4x the crops per batch
# at ~the same wall time (the sequential sample-level LSTM scan dominates), at
# the cost of 4x fewer optimizer steps/epoch; lr 2e-3 (sqrt-scaling) compensates.
# Same tuning as the diffssl coloration black-box run. --
BATCH_SIZE       = 64       # 02b parity value: 16 (dataset_la2a default)
LR               = 2e-3     # 02b parity value: 1e-3 (sqrt-scaled for the 4x batch)
MAX_EPOCHS       = 100      # fixed budget == cosine T_max
STEP_NUM_SAMPLES = 4410     # diffssl TBPTT sub-step (0.1 s) — 02b parity
SCHEDULER        = "cosine"
ETA_MIN          = 1e-6
USE_AMP          = True
CHECK_VAL_EVERY_N_EPOCH = 1

# -- model core (identical to the diffssl color-bb run, except the 2 LA2A knobs) --
HIDDEN_SIZE     = 32
NUM_LAYERS      = 1
NUM_CONTROLS    = 2      # LA2A: [comp_limit, peak_reduction]
TVCOND_DIM      = 16
COND_BLOCK_SIZE = 128
COND_NUM_LAYERS = 1

# -- loss: EXACT 02b recipe. The input signal must stay the ONLY difference vs
# the LA2A SOTA black-box (clean comparison + structural sidechain claim), so no
# extra loss terms — they would confound the input-swap effect. env is void
# anyway: the oracle-GR multiply pins the envelope. GR comes from the dataset's
# on-the-fly gain_reduction_db (oracle); predicted/const GR is evaluated by
# inference-time swap only, never trained on.
TD_WEIGHT      = 0.5      # L1
FD_WEIGHT      = 0.5      # MR-STFT
ENV_WEIGHT     = 0.0      # off — 02b parity (envelope already pinned by the multiply)
PE_WEIGHT      = 0.0      # off — 02b parity
MRSTFT_VARIANT = "sota"   # exact 02b resolutions; "extended" is ablation-only

RUN_TAG    = "la2a_lstm32_color_bb_b64lr2e3"
RESUME_RUN = None

In [ ]:
# -- 4. Preview split — percentage-based, temporal within each recording -
# Every (comp_limit, peak_reduction) setting appears in train/val/test on
# DISJOINT temporal regions of its recording: conditioning is fully learnable
# and the test set is unseen audio at known settings (the LA2A convention).

pairs = discover_la2a_pairs(DATA_ROOT)
preview = build_la2a_split_manifest(
    pairs, seed=SPLIT_SEED, sample_length=SAMPLE_LENGTH,
    train_frac=TRAIN_FRAC, val_frac=VAL_FRAC, test_frac=TEST_FRAC,
    crops_per_pair=CROPS_PER_PAIR,
)

print(f"Recordings : {len(preview.pairs)}")
print(f"Settings   : {len(preview.settings)} unique [comp_limit, peak_reduction]")
print(f"  comp/limit values : {sorted({s[0] for s in preview.settings})}")
print(f"  peak_reduction    : {sorted({s[1] for s in preview.settings})}")
print(f"Regions    : train[0,{TRAIN_FRAC}) val[{TRAIN_FRAC},{round(TRAIN_FRAC+VAL_FRAC,3)}) "
      f"test[{round(TRAIN_FRAC+VAL_FRAC,3)},1) of every recording")
print(f"Crop totals: {preview.crop_counts}  (<= {CROPS_PER_PAIR} per recording)")
mins = {k: v * SAMPLE_LENGTH / SAMPLE_RATE / 60 for k, v in preview.crop_counts.items()}
print("Audio (min): " + "  ".join(f"{k}={mins[k]:.1f}" for k in ("train", "val", "test")))

assert all(preview.crop_counts[k] > 0 for k in ("train", "val", "test")), "empty split!"

In [ ]:
# -- 5. Model size ----------------------------------------------------

model = ColorBlackboxLSTM(
    num_controls=NUM_CONTROLS, hidden_size=HIDDEN_SIZE, num_layers=NUM_LAYERS,
    tvcond_dim=TVCOND_DIM, cond_block_size=COND_BLOCK_SIZE, cond_num_layers=COND_NUM_LAYERS,
)
n_params = sum(p.numel() for p in model.parameters())
print(f"ColorBlackboxLSTM: {n_params:,} params  "
      f"(hidden={HIDDEN_SIZE}, tvcond_dim={TVCOND_DIM}, controls={NUM_CONTROLS})")
for name, mod in model.named_children():
    print(f"  {name:10s} {sum(p.numel() for p in mod.parameters()):,}")
print(f"\nDiffssl color-bb run was 8,033 (4 knobs); LA2A has 2 knobs -> {n_params:,}. "
      f"LA2A SOTA LSTM32TVC is 7,905 (same trunk, dry input) — parameter-matched.")
print(f"Crop {SAMPLE_LENGTH} ({SAMPLE_LENGTH/SAMPLE_RATE:.2f}s) | {SAMPLE_RATE} Hz | "
      f"TBPTT step {STEP_NUM_SAMPLES} | tvcond block {COND_BLOCK_SIZE}")

In [ ]:
# -- 6. DataModule + amp-match baseline --------------------------------
# NO zero-init anchor here (that's the point): the untrained model is a
# randomly-initialised readout, NOT the amplitude match. Print both L1s on a
# real batch — the amp-match number is the baseline training must beat.

torch.backends.cudnn.benchmark = True
torch.set_float32_matmul_precision("high")

assert DATA_ROOT.startswith("/content/"), "Run the cache cell first (cell 2)."

NUM_WORKERS = min(8, os.cpu_count() or 2)
print(f"DataLoader num_workers: {NUM_WORKERS}")

if RESUME_RUN:
    RUN_NAME = RESUME_RUN
    RUN_DIR = os.path.join(OUTPUT_DIR, RUN_NAME)
    _resume_ckpt = os.path.join(RUN_DIR, "checkpoints", "last.ckpt")
    print(f"RESUMING: {RUN_NAME}")
else:
    RUN_NAME = f"la2a_color_bb_{datetime.now():%Y%m%d_%H%M%S}_{RUN_TAG}"
    RUN_DIR = os.path.join(OUTPUT_DIR, RUN_NAME)
    _resume_ckpt = None
    print(f"NEW run: {RUN_NAME}")

os.makedirs(RUN_DIR, exist_ok=True)
split_path = os.path.join(RUN_DIR, "split_manifest.json")

dm = La2aCropDataModule(
    data_root=DATA_ROOT, sample_length=SAMPLE_LENGTH, sample_rate=SAMPLE_RATE,
    batch_size=BATCH_SIZE, split_seed=SPLIT_SEED,
    train_frac=TRAIN_FRAC, val_frac=VAL_FRAC, test_frac=TEST_FRAC,
    crops_per_pair=CROPS_PER_PAIR, rms_window=RMS_WINDOW,
    split_manifest_path=split_path, num_workers=NUM_WORKERS,
)
dm.setup()
print(f"Train/val/test crops: {len(dm.train_dataset)} / {len(dm.val_dataset)} / {len(dm.test_dataset)}")
print(f"Batches/epoch (train): {len(dm.train_dataloader())}  (batch_size={BATCH_SIZE})")

# -- baselines on a real val batch --------------------------------------
from amplitude_match import amplitude_match

if not RESUME_RUN:
    _dry, _gr, _wet, _p = next(iter(dm.val_dataloader()))
    with torch.no_grad():
        model.reset_states()
        _y0 = model(_dry, _gr, _p)
    _l1_untrained = float(torch.nn.functional.l1_loss(_y0, _wet))
    _l1_amp = float(torch.nn.functional.l1_loss(amplitude_match(_dry, _gr), _wet))
    print(f"untrained model crop L1 vs wet : {_l1_untrained:.6f}  (random readout)")
    print(f"amp-match      crop L1 vs wet : {_l1_amp:.6f}  <- the baseline to beat")
    model.reset_states()
    del _dry, _gr, _wet, _p, _y0

In [ ]:
# -- 6b. GR range vs the [-30, +5] dB clamp — LA2A-specific check --------
# The model clamps GR to [GR_DB_MIN, GR_DB_MAX] = [-30, +5] dB before the
# multiply (amplitude_match convention, justified on DIFFSSL where the exported
# curves span ~[-28.3, +4.2] dB). LA2A is NOT automatically inside that range,
# so this cell re-verifies it on the actual split crops. Full-file reference
# scan of all 84 recordings (local, 2026-07-29): energy-valid GR (dry RMS >
# -60 dB) spans [-49.6, +36.2] dB, but the clamp touches almost no real
# compression:
#   below -30 dB : 0.009% of valid samples — deep-GR tails, pr >= 85 only
#                  (worst single recording 0.21%, id 263 cl=1 pr=100)
#   above  +5 dB : 0.45% of valid samples — wet noise floor / capture-session
#                  level mismatch, NOT compression (present even at pr=0 where
#                  nothing compresses; ids 221+ are the worst) — the clamp is
#                  protective there, exactly the diffssl silence-tail rationale.
# Keeping [-30, +5] is also required for cascade consistency: the LA2A GR
# predictor's targets are normalised over the same [-30, +5] (gr_target.py),
# so predicted GR can never leave this range — widening the clamp only here
# would create a train/deploy mismatch. The asserts below fail if a dataset /
# split change ever pushes materially more valid mass outside the clamp.

from itertools import chain

from amplitude_match import GR_DB_MAX, GR_DB_MIN
from src.dsp_torch import windowed_rms

ENERGY_FLOOR_DB = -60.0

_n = _lo = _hi = _v = _vlo = _vhi = 0
_gmin, _gmax = float("inf"), float("-inf")
_vmin, _vmax = float("inf"), float("-inf")
for _dry, _gr, _wet, _p in chain(dm.train_dataloader(), dm.val_dataloader(),
                                 dm.test_dataloader()):
    _gmin = min(_gmin, float(_gr.min())); _gmax = max(_gmax, float(_gr.max()))
    _n += _gr.numel()
    _lo += int((_gr < GR_DB_MIN).sum()); _hi += int((_gr > GR_DB_MAX).sum())
    _mask = 20 * torch.log10(windowed_rms(_dry, RMS_WINDOW)) > ENERGY_FLOOR_DB
    _gv = _gr[_mask]
    if _gv.numel():
        _vmin = min(_vmin, float(_gv.min())); _vmax = max(_vmax, float(_gv.max()))
        _v += _gv.numel()
        _vlo += int((_gv < GR_DB_MIN).sum()); _vhi += int((_gv > GR_DB_MAX).sum())

_vlo_pct, _vhi_pct = 100 * _vlo / max(_v, 1), 100 * _vhi / max(_v, 1)
print(f"clamp range            : [{GR_DB_MIN}, {GR_DB_MAX}] dB")
print(f"GR span (all crops)    : [{_gmin:.2f}, {_gmax:.2f}] dB")
print(f"GR span (energy-valid) : [{_vmin:.2f}, {_vmax:.2f}] dB  (dry RMS > {ENERGY_FLOOR_DB} dB)")
print(f"clipped (all)          : below {100*_lo/_n:.4f}%  above {100*_hi/_n:.4f}%")
print(f"clipped (energy-valid) : below {_vlo_pct:.4f}%  above {_vhi_pct:.4f}%")

assert _vlo_pct < 0.1, (
    f"{_vlo_pct:.3f}% of energy-valid GR samples below {GR_DB_MIN} dB (expected "
    "~0.01%): deep compression is being materially clipped — re-run the full-file "
    "scan and reconsider the clamp before training."
)
assert _vhi_pct < 2.0, (
    f"{_vhi_pct:.3f}% of energy-valid GR samples above {GR_DB_MAX} dB (expected "
    "~0.5%, noise-floor/session-mismatch artefacts): check the dataset before training."
)
print("OK: clamp only touches deep-GR tails (<0.1%) and noise-floor positives (<2%).")


In [ ]:
# -- 7. Train ---------------------------------------------------------

with open(os.path.join(RUN_DIR, "hparams.json"), "w") as f:
    json.dump({
        "approach": "color_bb: y = tanh(lin(LSTM(x * 10^(gr/20), tvcond))) - SOTA trunk "
                    "on the amplitude-matched signal; gain by explicit multiply",
        "model_type": "ColorBlackboxLSTM",
        "model_ref": "06_output/train_lstm_color_blackbox retargeted to LA2A "
                     "(same trunk as train_lstm_la2a_tvc, matched input instead of dry)",
        "dataset": "SignalTrain-LA2A",
        "setting": "multi (all 42 comp_limit x peak_reduction settings; tvcond on 2 knobs)",
        "conditioning": "knobs via tvcond (TVFiLMCond); GR only via the input multiply "
                        "(model never sees the GR value)",
        "gr_source": "on-the-fly gain_reduction_db(dry, wet, 1024) == exported gr_curves/*.pt "
                     "(oracle); predicted/const GR is inference-time swap only, never trained on",
        "sample_rate": SAMPLE_RATE, "sample_length": SAMPLE_LENGTH, "batch_size": BATCH_SIZE,
        "rms_window": RMS_WINDOW, "step_num_samples": STEP_NUM_SAMPLES,
        "regime_deviation": "SPEED-TUNED vs 02b: batch 64 @ lr 2e-3 (02b: 16 @ 1e-3; "
                            "sqrt-scaled LR) -> 4x fewer optimizer steps/epoch; model, "
                            "loss, split, sub-step and epoch budget unchanged (same "
                            "tuning as the diffssl color-bb run)",
        "param_order": LA2A_PARAM_ORDER, "param_ranges": LA2A_PARAM_RANGES,
        "split_seed": SPLIT_SEED,
        "split_policy": "temporal_within_recording (every setting in train/val/test; test = unseen audio regions)",
        "split_fracs": {"train": TRAIN_FRAC, "val": VAL_FRAC, "test": TEST_FRAC},
        "crops_per_pair": CROPS_PER_PAIR,
        "num_settings": len(dm.split.settings), "num_recordings": len(dm.split.pairs),
        "crop_counts": dm.split.crop_counts,
        "model": {"hidden_size": HIDDEN_SIZE, "num_layers": NUM_LAYERS,
                   "num_controls": NUM_CONTROLS, "tvcond_dim": TVCOND_DIM,
                   "cond_block_size": COND_BLOCK_SIZE, "cond_num_layers": COND_NUM_LAYERS,
                   "num_params": n_params},
        "loss": {"td_weight": TD_WEIGHT, "fd_weight": FD_WEIGHT,
                 "env_weight": ENV_WEIGHT, "pe_weight": PE_WEIGHT,
                 "mrstft_variant": MRSTFT_VARIANT,
                 "kind": "td*L1 + fd*MRSTFT + env*envdB_L1 + pe*preemph_L1",
                 "policy": "exact 02b recipe (env=pe=0, mrstft='sota'); input swap is the "
                           "only MODEL/LOSS diff vs the LA2A SOTA black-box (see "
                           "regime_deviation for the batch/lr speed change)"},
        "metrics": ["esr", "rmse", "mae", "mse"],
        "optimizer": f"adamw + {SCHEDULER}",
        "scheduler": SCHEDULER, "eta_min": ETA_MIN, "use_amp": USE_AMP,
        "check_val_every_n_epoch": CHECK_VAL_EVERY_N_EPOCH,
        "training": "la2a_crop_batches + tbptt_substeps (reset each batch)",
        "lr": LR, "max_epochs": MAX_EPOCHS,
    }, f, indent=2)

system = GainPriorSystem(
    model=model, lr=LR, step_num_samples=STEP_NUM_SAMPLES,
    td_weight=TD_WEIGHT, fd_weight=FD_WEIGHT,
    env_weight=ENV_WEIGHT, pe_weight=PE_WEIGHT, mrstft_variant=MRSTFT_VARIANT,
    scheduler=SCHEDULER, max_epochs=MAX_EPOCHS, eta_min=ETA_MIN, use_amp=USE_AMP,
)

ckpt_dir = os.path.join(RUN_DIR, "checkpoints")
callbacks = [
    ModelCheckpoint(dirpath=ckpt_dir, monitor="loss/val", mode="min", save_top_k=3,
                    save_last=True, filename="best-{epoch:03d}-{step}",
                    auto_insert_metric_name=False),
    LearningRateMonitor(logging_interval="epoch"),
    TQDMProgressBar(refresh_rate=10),
]
loggers = [
    TensorBoardLogger(save_dir=RUN_DIR, name="tb", version=""),
    CSVLogger(save_dir=RUN_DIR, name="csv", version=""),
]

trainer = pl.Trainer(
    max_epochs=MAX_EPOCHS, accelerator="gpu", devices=1,
    callbacks=callbacks, logger=loggers, log_every_n_steps=10,
    check_val_every_n_epoch=CHECK_VAL_EVERY_N_EPOCH,
)
trainer.fit(system, dm, ckpt_path=_resume_ckpt)
print(f"Best val loss: {callbacks[0].best_model_score:.6f}")
print(f"Best ckpt    : {callbacks[0].best_model_path}")

In [ ]:
# -- 8. Test (held-out temporal regions - unseen audio at known settings) --

best_ckpt = callbacks[0].best_model_path or os.path.join(ckpt_dir, "last.ckpt")
print(f"Testing with: {best_ckpt}")
trainer.test(system, datamodule=dm, ckpt_path=best_ckpt)

In [ ]:
# -- 9. Plot: prediction vs target + coloration share -------------------
# For each example: (top) waveform overlay, (bottom) the GR input driving the
# multiply. mean|color| = mean |y - matched|, the total learned coloration.
# POLARITY CHECK first: the tanh readout has learned -wet before (diffssl
# color-bb + gr_tfilm runs) — corr(pred, wet) must be strongly POSITIVE.

import matplotlib.pyplot as plt
import numpy as np
from system_gainprior import esr_metric

best = torch.load(callbacks[0].best_model_path, map_location="cuda", weights_only=False)
system.load_state_dict(best["state_dict"])
system.eval().cuda()
print(f"Loaded best checkpoint: {callbacks[0].best_model_path}")

val_batches = list(dm.val_dataloader())
dry, gr, wet, params = val_batches[len(val_batches) // 2]

with torch.no_grad():
    system.model.reset_states()
    pred, delta_db, color, ws_res = system.model(
        dry.cuda(), gr.cuda(), params.cuda(), return_parts_full=True)
    pred, delta_db = pred.cpu(), delta_db.cpu()
    color, ws_res = color.cpu(), ws_res.cpu()

# -- polarity check (see project memory: tanh-readout runs have inverted) ---
_pf, _wf = pred.flatten(), wet.flatten()
_corr = float((_pf * _wf).sum() / (_pf.norm() * _wf.norm() + 1e-12))
print(f"corr(pred, wet) = {_corr:+.4f}")
assert _corr > 0.5, (
    f"POLARITY INVERSION suspected (corr={_corr:+.4f}): the readout may have "
    "learned -wet. Do NOT trust td/fd metrics; negate the readout (signfix) "
    "before eval — see project_color_bb_polarity."
)

dry_np, wet_np, pred_np = dry.numpy(), wet.numpy(), pred.numpy()
gr_np = gr.numpy()
color_np = color.numpy()
n_plots = min(3, dry_np.shape[0])
fig, axes = plt.subplots(2 * n_plots, 1, figsize=(14, 4.6 * n_plots), squeeze=False)
t = np.arange(wet_np.shape[-1]) / SAMPLE_RATE
for r in range(n_plots):
    ax = axes[2 * r, 0]
    ax.plot(t, dry_np[r, 0], label="Dry", alpha=0.35, lw=0.5, color="gray")
    ax.plot(t, wet_np[r, 0], label="Target (wet)", alpha=0.8, lw=0.5)
    ax.plot(t, pred_np[r, 0], label="Predicted", alpha=0.8, lw=0.5)
    pv = torch.from_numpy(pred_np[r]); tv = torch.from_numpy(wet_np[r])
    mae_r = float(np.mean(np.abs(pred_np[r, 0] - wet_np[r, 0])))
    ax.set_title(f"crop {r} - MAE {mae_r:.4f} | ESR {float(esr_metric(tv, pv)):.4f} | "
                 f"mean|color| {np.abs(color_np[r]).mean():.5f}")
    ax.set_ylabel("amp"); ax.legend(loc="lower right", fontsize=8); ax.set_ylim(-1.05, 1.05)

    ax = axes[2 * r + 1, 0]
    ax.plot(t, gr_np[r, 0], label="GR input (dB) - applied verbatim", lw=0.7,
            color="#1f77b4")
    ax.set_ylabel("gain (dB)"); ax.legend(loc="lower right", fontsize=8)
axes[-1, 0].set_xlabel("Time (s)")
fig.suptitle(f"LA2A coloration black-box LSTM - best val loss {callbacks[0].best_model_score:.6f}", y=1.002)
fig.tight_layout()
plot_path = os.path.join(RUN_DIR, "eval_output_comparison.png")
fig.savefig(plot_path, dpi=150, bbox_inches="tight")
print(f"Saved plot -> {plot_path}")
plt.show()

In [ ]:
# -- 10. Learned coloration spectra ------------------------------------
# Is the matched->wet residual actually being synthesised? Compare PSDs of the
# TARGET coloration (wet - matched) and the LEARNED coloration (pred - matched)
# on the plotted val batch. Overlap is the win condition. NB: on LA2A the
# low-pr residual floor is partly a data artefact (capture-session mismatch,
# see project_la2a_session_mismatch) — judge trends, not absolute floors.

from scipy.signal import welch

matched_v = (dry * torch.pow(10.0, gr.clamp(-30.0, 5.0) / 20.0)).numpy()
tgt_col = (wet_np - matched_v)[:, 0].reshape(-1)
lrn_col = (pred_np - matched_v)[:, 0].reshape(-1)
wet_flat = wet_np[:, 0].reshape(-1)

fig, ax = plt.subplots(figsize=(11, 4.5))
for sig, name, style in ((wet_flat, "wet", dict(color="k", lw=1.2)),
                         (tgt_col, "target coloration (wet - matched)",
                          dict(color="#1f77b4", lw=1.2)),
                         (lrn_col, "learned coloration (pred - matched)",
                          dict(color="#d62728", lw=1.0))):
    f, Pxx = welch(sig, fs=SAMPLE_RATE, nperseg=8192)
    ax.semilogx(f, 10 * np.log10(Pxx + 1e-18), label=name, **style)
ax.set_xlabel("frequency (Hz)"); ax.set_ylabel("PSD (dB/Hz)")
rms = lambda v: float(np.sqrt(np.mean(v ** 2)))
ax.set_title(f"Coloration spectra - target {rms(tgt_col)/rms(wet_flat):.1%} of wet RMS, "
             f"learned {rms(lrn_col)/rms(wet_flat):.1%}")
ax.grid(alpha=0.3, which="both"); ax.legend(); ax.set_xlim(20, SAMPLE_RATE / 2)
fig.tight_layout()
spec_path = os.path.join(RUN_DIR, "eval_coloration_spectra.png")
fig.savefig(spec_path, dpi=150, bbox_inches="tight")
print(f"Saved plot -> {spec_path}")
plt.show()

In [ ]:
%load_ext tensorboard
%tensorboard --logdir "{RUN_DIR}/tb"